# Observer.ai: Kev vs Jev vs Laya as trace samplers

Scores the same held-out traces with each decision model and compares them as samplers: given a small review budget, which model puts the most wrong answers in front of a human?

**Models**
- **Kev, fine-tuned**: the run from `observer_kev_lora_finetune.ipynb`, served locally.
- **Jev**: TypeSafe's hosted model, called zero-shot over its API. It cannot be fine-tuned.
- **Laya, zero-shot**: the released checkpoint, served locally. Swap in the fine-tuned run from `observer_laya_finetune.ipynb` once it exists (one line in `MODELS`).
- **Heuristic**: the no-model rules from the other notebooks, as the bar every model has to clear.

**What is held equal:** the traces, the slice text (`build_state`), the two questions, the test portion and the metric code are the ones the fine-tuning notebooks use. **What is not:** Kev was trained on this task and the other two were not, and Jev's latency includes a network round trip. Read the tables with that in mind.

**Flow**
1. Load the traces, rebuild the same split, and take the same calibration and test portions
2. Check which models are reachable; a model that is not is skipped, not an error
3. Score the test portion (and the calibration portion) with each model
4. Sampler metrics at realistic base rates, one table for every model and signal
5. Head to head: is one model's lead bigger than the noise?
6. Where they differ: by failure type, and how much their picks overlap
7. Calibration, latency and cost

In [1]:
import hashlib, itertools, json, math, os, random, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---- config ----
TRACES_PATH  = Path("data/eval/labeled_traces.jsonl")
WORK_DIR     = Path("runs/observer-comparison").resolve()
KEV_MANIFEST = Path("runs/observer-kev-0.8b-ft/manifest.json")   # used to check this notebook rebuilds the same split
SAMPLE_RATE  = 0.05                                 # the headline budget
SAMPLE_RATES = [0.05, 0.10, 0.20]                   # every budget in the results table
TARGET_BASE_RATES = [0.10, 0.20]                    # share of wrong traces in the rebalanced test sets
N_RESAMPLES  = 20                                   # rebalanced test sets per base rate (seeds SEED, SEED+1, ...)
N_BOOTSTRAP  = 2000                                 # resamples of the test portion for confidence intervals
HELDOUT_FRAC = 0.2                                  # only used when the dataset has no `split` field
CALIB_FRAC   = 0.5                                  # share of held-out questions used to fit temperatures
SEED         = 7                                    # same seed as the fine-tuning notebooks: same split

DECISION_SIGNAL = "keep_or_drop"                    # the signal the summary and head-to-head use: keep_or_drop | answer_wrong
SUCCESS_MARGIN  = 0.10                              # the project's bar: beat the best heuristic by this much normalized recall
CALIBRATE       = True                              # also score the calibration portion, to fit a temperature per model.
                                                    # Doubles the number of Jev calls.

# One entry per model. Every model speaks the /v1/systemone protocol.
#   url, model    where to send requests and the model name to put in them
#   api_key_env   environment variable holding a bearer token (hosted models)
#   body          extra request fields
#   cache         keep answers on disk and reuse them. Only for models that cannot change under the same
#                 name and cost money per call. A local server is never cached: retraining would go unnoticed.
MODELS = {
    "kev_finetuned": {
        "label": "Kev 0.8B, fine-tuned",
        "url": "http://127.0.0.1:8010/v1/systemone", "model": "kev-latest",
    },
    "jev": {
        "label": "Jev, zero-shot (hosted)",
        "url": "https://api.typesafe.ai/v1/systemone", "model": "jev-latest",
        "api_key_env": "JEV_API_KEY", "cache": True,
    },
    "laya_zero_shot": {
        "label": "Laya multilingual, zero-shot",
        "url": "http://127.0.0.1:8011/v1/systemone", "model": "observer-base",
        # Laya cuts the end of a state that does not fit its token window, and our slice ends with the
        # final answer. 2048 fits every slice; a shorter state costs nothing extra.
        "body": {"max_len": 2048},
    },
    # Once observer_laya_finetune.ipynb has produced a run, serve it on 8012 and replace the entry above with:
    # "laya_finetuned": {"label": "Laya multilingual, fine-tuned",
    #                    "url": "http://127.0.0.1:8012/v1/systemone", "model": "observer-ft", "body": {"max_len": 2048}},
}

random.seed(SEED)
np.random.seed(SEED)
(WORK_DIR / "cache").mkdir(parents=True, exist_ok=True)

## 1. Load labeled traces

Expected schema, one JSON object per line (produced by your ReAct agent + gold-answer labeller):

```json
{
  "trace_id": "7f3a91c2e8",
  "question": "Which river flows through the capital of France?",
  "tool_calls": [{"name": "search", "args": "capital of France", "status": "ok"}],
  "evidence": ["The Eiffel Tower is in Paris, France."],
  "final_answer": "The Rhône flows through the capital, Lyon.",
  "label": "wrong"
}
```

`label` is one of `correct`, `partial`, `wrong`.

In [2]:
def load_traces(path):
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    bad = [r["trace_id"] for r in rows if r["label"] not in {"correct", "partial", "wrong"}]
    assert not bad, f"unknown labels on {len(bad)} traces, e.g. {bad[:3]}"
    return rows

traces = load_traces(TRACES_PATH)
labels = [t["label"] for t in traces]
print(f"{len(traces)} traces:", {k: labels.count(k) for k in ("correct", "partial", "wrong")})

4499 traces: {'correct': 1729, 'partial': 625, 'wrong': 2145}


## 2. Build trace slices

The `state` is the slice the Go engine will send: question, tool calls, top evidence and the final answer.

`build_state` is identical to the fine-tuning notebooks' and to the Go engine's `BuildState`, so every model reads the same text.

In [3]:
MAX_EVIDENCE = 3
MAX_EVIDENCE_CHARS = 1500

def build_state(t):
    tools = "\n".join(
        f"{i}. {c['name']}({c.get('args', '')!r}) -> {c.get('status', 'ok')}"
        for i, c in enumerate(t.get("tool_calls", []), 1)
    ) or "(none)"
    evidence = "\n".join(
        f"[{i}] {e[:MAX_EVIDENCE_CHARS]}" for i, e in enumerate(t.get("evidence", [])[:MAX_EVIDENCE], 1)
    ) or "(none)"
    return (
        f"USER QUESTION:\n{t['question']}\n\n"
        f"TOOL CALLS:\n{tools}\n\n"
        f"EVIDENCE:\n{evidence}\n\n"
        f"FINAL ANSWER:\n{t['final_answer']}"
    )

print(build_state(traces[0]))

USER QUESTION:
What was the estimated population of the city that Luis Villalta Aquino was from in 2012?

TOOL CALLS:
1. search({'query': 'Luis Villalta Aquino city 2012 population'}) -> ok

EVIDENCE:
[1] Luis Villalta Aquino (October 2, 1969, Lima, Peru – March 3, 2004, Pompano Beach, Florida) was a professional boxer, who was nicknamed "El Puma" during his career.
[2] San Luis Potosí, commonly called SLP or simply San Luis, is the capital and the most populous city of the Mexican state of San Luis Potosí. The city lies at an elevation of 1850 m . It has an estimated population of 735,886 in the city proper and a population of approximately 1,021,688 in its metropolitan area, formed with the neighbour city of Soledad de Graciano Sánchez and some other small townships inside the urban area, which makes the metropolitan area of Greater San Luis Potosí the eleventh largest in Mexico.
[3] Pompano Beach is a city in Broward County, Florida, United States, along the coast of the Atlantic Oc

## 3. Questions and the split

The two questions, the split by question and the calibration and test portions are rebuilt with the fine-tuning notebooks' code and seed, then checked against the Kev run's manifest. Kev was trained on the train split only, so the test portion is unseen by every model here.

Both signals are scored for every model. Partial answers count as wrong.
- `answer_wrong` (noul): `p(answer_wrong)`
- `keep_or_drop` (choice): `p(keep)`, the probability of the `keep` option

In [4]:
QUESTIONS = {
    "answer_wrong": {
        "type": "noul",
        "instructions": "Is the final answer wrong or only partially correct for the user's question?",
    },
    "keep_or_drop": {
        "type": "choice",
        "instructions": "Should this agent trace be kept for human review?",
        "criteria": {
            "keep": "The final answer is wrong, only partly correct or unsupported by the evidence, or the agent misused or skipped its tools.",
            "drop": "The final answer is correct, complete and supported by the evidence, and the agent used its tools appropriately.",
        },
    },
}

def is_wrong(t):
    return t["label"] != "correct"

def split_traces(rows):
    """Split by question, so natural and seeded runs of one question stay on the same side."""
    if all(t.get("split") is not None for t in rows):
        unknown = {t["split"] for t in rows} - {"train", "heldout"}
        assert not unknown, f"unknown split values: {sorted(unknown)}"
        method = "dataset_split_field"
        train = [t for t in rows if t["split"] == "train"]
        heldout = [t for t in rows if t["split"] == "heldout"]
    elif all(t.get("qid") is not None for t in rows):
        method = f"group_by_qid(heldout_frac={HELDOUT_FRAC}, seed={SEED})"
        qids = sorted({t["qid"] for t in rows})
        random.Random(SEED).shuffle(qids)
        heldout_qids = set(qids[:round(len(qids) * HELDOUT_FRAC)])
        train = [t for t in rows if t["qid"] not in heldout_qids]
        heldout = [t for t in rows if t["qid"] in heldout_qids]
    else:
        raise ValueError(
            "cannot split by question: the dataset needs `qid` or `split` on every trace. "
            "Re-run observer_eval_dataset_prep.ipynb so labeled_traces.jsonl carries one of them."
        )
    random.Random(SEED).shuffle(train)
    return train, heldout, method

def question_key(t):
    return t["qid"] if t.get("qid") is not None else t["question"]

train, heldout, SPLIT_METHOD = split_traces(traces)
assert train and heldout, f"empty split: train {len(train)}, held-out {len(heldout)}"

leaked = {question_key(t) for t in train} & {question_key(t) for t in heldout}
assert not leaked, f"{len(leaked)} questions appear in both train and held-out, e.g. {sorted(leaked)[:3]}"

def split_by_question(rows, frac, seed):
    keys = sorted({question_key(t) for t in rows})
    random.Random(seed).shuffle(keys)
    first = set(keys[:round(len(keys) * frac)])
    return [t for t in rows if question_key(t) in first], [t for t in rows if question_key(t) not in first]

calib, test = split_by_question(heldout, CALIB_FRAC, SEED)
assert not {question_key(t) for t in calib} & {question_key(t) for t in test}, "a question is in both calibration and test"

y_calib = np.array([is_wrong(t) for t in calib])
y_test  = np.array([is_wrong(t) for t in test])
for name, rows in (("calibration", calib), ("test", test)):
    n_q = len({question_key(t) for t in rows})
    print(f"{name:<12}{len(rows):>5} traces, {n_q:>5} unique questions, {np.mean([is_wrong(t) for t in rows]):.1%} wrong or partial")

# The fine-tuning notebook recorded the split it trained and tested on. A mismatch means the numbers
# here would not be comparable with it, or that Kev saw some of these traces in training.
if KEV_MANIFEST.exists():
    recorded = json.loads(KEV_MANIFEST.read_text())
    for name, rows in (("train", train), ("calibration", calib), ("test", test)):
        want = recorded["split"][name]["traces"]
        assert len(rows) == want, f"{name}: {len(rows)} traces here, {want} in the Kev run's manifest"
    assert recorded["questions"] == QUESTIONS, "the questions differ from the ones the Kev run was trained on"
    print("split and questions match the Kev run's manifest")
else:
    print(f"no manifest at {KEV_MANIFEST}; the split was not cross-checked")

calibration   444 traces,   300 unique questions, 62.6% wrong or partial
test          456 traces,   300 unique questions, 57.5% wrong or partial
split and questions match the Kev run's manifest


## 4. The models

Every model is asked the same thing in the same way: one request per trace, carrying the slice and both questions.

- **Kev** needs its server on port 8010 serving the fine-tuned run: `uv run --extra serve python -m kev.serve --run run --port 8010` in the Kev repo.
- **Jev** needs `JEV_API_KEY` in the environment of this notebook's kernel. Calls go to TypeSafe's servers and are billed to that key. The traces here are built from HotpotQA, a public dataset; think before pointing this at private traces.
- **Laya** needs its server on port 8011. `observer_laya_finetune.ipynb` prints the exact command.

This cell sends one trace to each model. A model that does not answer is left out of everything below, with the reason printed, so the notebook still runs with the models you have.

In [5]:
SIGNAL_READ = {
    "answer_wrong": lambda a: a["noul"],
    "keep_or_drop": lambda a: a["probabilities"]["keep"],
}
assert DECISION_SIGNAL in SIGNAL_READ, f"DECISION_SIGNAL must be one of {list(SIGNAL_READ)}"
RETRY_STATUS = {429, 500, 502, 503, 504, 529}   # rate limited or overloaded: wait and try again

def call_model(name, state, timeout=60, attempts=5):
    """One request. Returns the signals, the round-trip time of the successful attempt and the usage block."""
    cfg = MODELS[name]
    headers = {}
    if cfg.get("api_key_env"):
        key = os.environ.get(cfg["api_key_env"])
        if not key:
            raise RuntimeError(f"{cfg['api_key_env']} is not set in this kernel's environment")
        headers["Authorization"] = f"Bearer {key}"
    body = {"model": cfg["model"], "state": state, "questions": QUESTIONS, **cfg.get("body", {})}
    for attempt in range(attempts):
        t0 = time.perf_counter()
        r = requests.post(cfg["url"], json=body, headers=headers, timeout=timeout)
        latency_ms = (time.perf_counter() - t0) * 1000
        if r.status_code in RETRY_STATUS and attempt < attempts - 1:
            time.sleep(min(2 ** attempt, 30))
            continue
        if not r.ok:
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
        out = r.json()
        usage = out.get("usage") or {}
        return {
            **{signal: float(read(out["answers"][signal])) for signal, read in SIGNAL_READ.items()},
            "latency_ms": latency_ms,
            "input_tokens": usage.get("input_tokens"),
            "truncated": bool(usage.get("truncated")),
            "served_model": out.get("model"),
        }

AVAILABLE, SKIPPED = [], {}
for name, cfg in MODELS.items():
    try:
        first = call_model(name, build_state(test[0]), attempts=1)
        AVAILABLE.append(name)
        print(f"{name:<16} ok       {cfg['label']}; answers as {first['served_model']!r}, {first['latency_ms']:.0f} ms")
    except Exception as exc:
        SKIPPED[name] = f"{type(exc).__name__}: {exc}"[:160]
        print(f"{name:<16} SKIPPED  {SKIPPED[name]}")
assert AVAILABLE, "no model is reachable; start a server or set an API key, then run this cell again"

kev_finetuned    ok       Kev 0.8B, fine-tuned; answers as 'kev-latest', 5567 ms
jev              ok       Jev, zero-shot (hosted); answers as 'jev-1.13.0', 954 ms
laya_zero_shot   SKIPPED  ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8011): Max retries exceeded with url: /v1/systemone (Caused by NewConnectionError("HTTPConnection(hos


## 5. Score the traces

Scores the test portion with every reachable model, and the calibration portion too when `CALIBRATE` is on. Each call's round-trip time is kept for the latency table.

Answers from a model marked `cache` are stored in `runs/observer-comparison/cache/` and reused on the next run, so re-running the notebook does not pay for the same Jev calls twice. The cache key covers the model name, the questions and the slice text.

In [6]:
def cache_key(name, state):
    cfg = MODELS[name]
    payload = json.dumps([cfg["url"], cfg["model"], cfg.get("body", {}), QUESTIONS, state], sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()

def load_cache(name):
    path = WORK_DIR / "cache" / f"{name}.jsonl"
    if not MODELS[name].get("cache") or not path.exists():
        return {}
    return {row["key"]: row for row in map(json.loads, path.read_text().splitlines()) if row}

def score_model(name, rows):
    """Score rows with one model. Returns one record per row, in order."""
    cached = load_cache(name)
    path = WORK_DIR / "cache" / f"{name}.jsonl"
    records, reused = [], 0
    with open(path, "a") if MODELS[name].get("cache") else open(os.devnull, "w") as store:
        for i, t in enumerate(rows, 1):
            state = build_state(t)
            key = cache_key(name, state)
            if key in cached:
                record, reused = cached[key], reused + 1
            else:
                record = {"key": key, "trace_id": t["trace_id"], **call_model(name, state)}
                store.write(json.dumps(record) + "\n")
                store.flush()
            records.append(record)
            if i % 100 == 0 or i == len(rows):
                print(f"  {name}: {i}/{len(rows)}" + (f" ({reused} from cache)" if reused else ""), flush=True)
    return records

portions = {"test": test, **({"calibration": calib} if CALIBRATE else {})}
SCORES = {}    # model -> portion -> list of records
for name in AVAILABLE:
    SCORES[name] = {portion: score_model(name, rows) for portion, rows in portions.items()}
    cut = sum(r["truncated"] for records in SCORES[name].values() for r in records)
    if cut:
        print(f"  WARNING: {name} cut {cut} states short; raise max_len in its MODELS entry")

def signal_of(name, signal, portion="test"):
    return np.array([r[signal] for r in SCORES[name][portion]])

  kev_finetuned: 100/456
  kev_finetuned: 200/456
  kev_finetuned: 300/456
  kev_finetuned: 400/456
  kev_finetuned: 456/456
  kev_finetuned: 100/444
  kev_finetuned: 200/444
  kev_finetuned: 300/444
  kev_finetuned: 400/444
  kev_finetuned: 444/444
  jev: 100/456 (100 from cache)
  jev: 200/456 (200 from cache)
  jev: 300/456 (300 from cache)
  jev: 400/456 (400 from cache)
  jev: 456/456 (456 from cache)
  jev: 100/444 (100 from cache)
  jev: 200/444 (200 from cache)
  jev: 300/444 (300 from cache)
  jev: 400/444 (400 from cache)
  jev: 444/444 (444 from cache)


## 6. Sampler metrics

**Normalized recall** = (sampler − random) / (oracle − random), at a fixed sample budget.
- sampler: share of wrong answers that land in the top-k by the signal
- random: expected share caught by random sampling (= the sample rate)
- oracle: best possible with gold labels = min(1, k / number of wrong answers)

**Realistic base rates.** The test portion is about 60% wrong or partial, far above production failure rates, which makes a 5% budget too easy to fill. Sampling metrics are therefore measured on rebalanced test sets: every correct trace is kept and the wrong ones are downsampled to each target base rate. This is repeated with `N_RESAMPLES` seeds and reported as mean and standard deviation. Every model is scored on the same rebalanced sets.

AUROC does not depend on the base rate, so it is computed once on the full test portion.

In [7]:
def sampler_metrics(p_wrong, y_wrong, rate=SAMPLE_RATE):
    y = np.asarray(y_wrong, dtype=bool)
    n, n_wrong = len(y), int(y.sum())
    k = max(1, math.ceil(rate * n))
    top_k = np.argsort(-np.asarray(p_wrong), kind="stable")[:k]   # ties keep the caller's (shuffled) order
    sampler = y[top_k].sum() / n_wrong
    random_r = k / n
    oracle = min(1.0, k / n_wrong)
    return {
        "recall_at_rate": sampler,
        "random_recall": random_r,
        "oracle_recall": oracle,
        "normalized_recall": (sampler - random_r) / (oracle - random_r),
    }

def rebalanced_indices(y, base_rate, seed):
    """Every correct trace plus a seeded sample of wrong ones, in shuffled order, so wrong traces make up `base_rate`."""
    rng = np.random.default_rng(seed)
    correct_idx, wrong_idx = np.flatnonzero(~y), np.flatnonzero(y)
    n_wrong = round(base_rate * len(correct_idx) / (1 - base_rate))
    assert 0 < n_wrong <= len(wrong_idx), f"cannot reach base rate {base_rate} with {len(wrong_idx)} wrong traces"
    return rng.permutation(np.concatenate([correct_idx, rng.choice(wrong_idx, n_wrong, replace=False)]))

RESAMPLES = {
    br: [rebalanced_indices(y_test, br, SEED + i) for i in range(N_RESAMPLES)]
    for br in TARGET_BASE_RATES
}
RESULTS = {}   # (model, signal) -> rows of the results table

def evaluate_signal(model, signal, p):
    """Add one model's rows to RESULTS: every base rate x sample rate, mean and std over the rebalanced test sets."""
    p = np.asarray(p, dtype=float)
    auroc = roc_auc_score(y_test, p)
    rows = []
    for br, resamples in RESAMPLES.items():
        for rate in SAMPLE_RATES:
            runs = [sampler_metrics(p[idx], y_test[idx], rate) for idx in resamples]
            col = lambda key: np.array([m[key] for m in runs])
            rows.append({
                "model": model, "signal": signal, "base_rate": br, "sample_rate": rate,
                "n_rebalanced": len(resamples[0]), "auroc_full_test": auroc,
                "recall_mean": col("recall_at_rate").mean(), "recall_std": col("recall_at_rate").std(),
                "oracle_recall": col("oracle_recall").mean(), "random_recall": col("random_recall").mean(),
                "norm_recall_mean": col("normalized_recall").mean(), "norm_recall_std": col("normalized_recall").std(),
            })
    RESULTS[(model, signal)] = rows
    return rows

def results_table(models=None):
    rows = [r for key, group in RESULTS.items() if models is None or key[0] in models for r in group]
    return pd.DataFrame(rows).sort_values(["base_rate", "sample_rate", "model", "signal"], kind="stable").reset_index(drop=True)

def headline(model, signal):
    """Normalized recall at the headline SAMPLE_RATE, per base rate."""
    return {r["base_rate"]: r["norm_recall_mean"] for r in RESULTS[(model, signal)] if r["sample_rate"] == SAMPLE_RATE}

def show(table):
    print(table.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

for name in AVAILABLE:
    for signal in SIGNAL_READ:
        evaluate_signal(name, signal, signal_of(name, signal))

for br, resamples in RESAMPLES.items():
    idx = resamples[0]
    print(f"base rate {br:.0%}: {len(idx)} traces per rebalanced test set, {int(y_test[idx].sum())} wrong, {N_RESAMPLES} seeds")

base rate 10%: 216 traces per rebalanced test set, 22 wrong, 20 seeds
base rate 20%: 242 traces per rebalanced test set, 48 wrong, 20 seeds


### Heuristic reference

Scores the test portion with simple rules and no model. It is the bar: a model that cannot beat a few `if` statements is not earning its latency.

Each rule is oriented so that a higher value is more suspicious:
- `short_answer`: fewer words in the final answer
- `few_evidence`: fewer evidence items
- `no_search`: `used_search` is false
- `tool_error`: any tool call with a status other than `ok`
- `bad_stop`: `stop_reason` other than `answered`
- `low_overlap`: share of the final answer's words that do not appear in the evidence text

The rules are combined with a logistic regression fitted on the train split, and each rule is also scored on its own.

In [8]:
def words(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def heuristic_features(t):
    answer_words = words(t["final_answer"])
    evidence_words = set(words(" ".join(t.get("evidence", []))))
    overlap = np.mean([w in evidence_words for w in answer_words]) if answer_words else 0.0
    tool_calls = t.get("tool_calls", [])
    return {
        "short_answer": -len(answer_words),
        "few_evidence": -len(t.get("evidence", [])),
        "no_search":    float(not t.get("used_search", any(c.get("name") == "search" for c in tool_calls))),
        "tool_error":   float(any(c.get("status", "ok") != "ok" for c in tool_calls)),
        "bad_stop":     float(t.get("stop_reason", "answered") != "answered"),
        "low_overlap":  1.0 - overlap,
    }

RULES = list(heuristic_features(train[0]))
X_train = np.array([[heuristic_features(t)[r] for r in RULES] for t in train], dtype=float)
X_test  = np.array([[heuristic_features(t)[r] for r in RULES] for t in test], dtype=float)
y_train = np.array([is_wrong(t) for t in train])

print(f"{'rule':<14}{'mean when correct':>20}{'mean when wrong':>18}   (test portion)")
for j, rule in enumerate(RULES):
    print(f"{rule:<14}{X_test[~y_test, j].mean():>20.3f}{X_test[y_test, j].mean():>18.3f}")

heuristic_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))
heuristic_model.fit(X_train, y_train)
evaluate_signal("heuristic", "logreg", heuristic_model.predict_proba(X_test)[:, 1])
for j, rule in enumerate(RULES):
    evaluate_signal("heuristic", rule, X_test[:, j])

best_rule = max(RULES, key=lambda rule: np.mean(list(headline("heuristic", rule).values())))
REFERENCE = {"heuristic_logreg": ("heuristic", "logreg"), f"heuristic_{best_rule}": ("heuristic", best_rule)}
print(f"heuristic reference rows: the logistic regression over all rules, and the best single rule ({best_rule})")

rule             mean when correct   mean when wrong   (test portion)
short_answer                -2.046            -6.412
few_evidence                -3.237            -2.969
no_search                    0.005             0.076
tool_error                   0.010             0.053
bad_stop                     0.000             0.004
low_overlap                  0.155             0.254
heuristic reference rows: the logistic regression over all rules, and the best single rule (few_evidence)


## 7. Summary

One row per model, read through `DECISION_SIGNAL`, plus the two heuristic reference rows.

- `auroc`: ranking quality on the full test portion, with a 95% bootstrap interval. 0.5 is chance.
- `norm_recall@5% (10%)` and `(20%)`: normalized recall at the 5% budget when 10% or 20% of traces are wrong, mean ± standard deviation over the rebalanced test sets. 0 is random sampling, 1 is the oracle.
- `beats_heuristic`: whether it clears the project's bar, the best heuristic plus `SUCCESS_MARGIN`, at every base rate.

In [9]:
rng = np.random.default_rng(SEED)
BOOT = rng.integers(0, len(y_test), size=(N_BOOTSTRAP, len(y_test)))
BOOT = BOOT[[0 < y_test[idx].sum() < len(idx) for idx in BOOT]]   # an AUROC needs both classes

def auroc_interval(p):
    values = np.array([roc_auc_score(y_test[idx], p[idx]) for idx in BOOT])
    return np.percentile(values, [2.5, 97.5])

def series_of(row_key):
    """The scores behind a summary row: a model's decision signal, or a heuristic reference."""
    if row_key in REFERENCE:
        model, signal = REFERENCE[row_key]
        scores = heuristic_model.predict_proba(X_test)[:, 1] if signal == "logreg" else X_test[:, RULES.index(signal)]
        return model, signal, scores
    return row_key, DECISION_SIGNAL, signal_of(row_key, DECISION_SIGNAL)

def by_rate(model, signal, rate=SAMPLE_RATE):
    return {r["base_rate"]: r for r in RESULTS[(model, signal)] if r["sample_rate"] == rate}

ROW_KEYS = AVAILABLE + list(REFERENCE)
bar = {br: max(by_rate(*REFERENCE[k])[br]["norm_recall_mean"] for k in REFERENCE) + SUCCESS_MARGIN for br in TARGET_BASE_RATES}

summary_rows = []
for key in ROW_KEYS:
    model, signal, scores = series_of(key)
    lo, hi = auroc_interval(scores)
    rates = by_rate(model, signal)
    row = {"model": key, "signal": signal, "auroc": roc_auc_score(y_test, scores), "auroc_lo": lo, "auroc_hi": hi}
    for br in TARGET_BASE_RATES:
        row[f"norm_recall@{SAMPLE_RATE:.0%} ({br:.0%})"] = rates[br]["norm_recall_mean"]
        row[f"std ({br:.0%})"] = rates[br]["norm_recall_std"]
    row["beats_heuristic"] = "" if key in REFERENCE else all(rates[br]["norm_recall_mean"] >= bar[br] for br in TARGET_BASE_RATES)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
print(f"AUROC is on the full test portion ({len(test)} traces, {y_test.mean():.1%} wrong), not on the rebalanced sets")
print("the bar to beat (best heuristic + margin): " + ", ".join(f"{bar[br]:.3f} at {br:.0%}" for br in TARGET_BASE_RATES))
if SKIPPED:
    print("not compared (unreachable): " + ", ".join(SKIPPED))
summary.style.format(precision=3).hide(axis="index")

AUROC is on the full test portion (456 traces, 57.5% wrong), not on the rebalanced sets
the bar to beat (best heuristic + margin): 0.685 at 10%, 0.913 at 20%
not compared (unreachable): laya_zero_shot


model,signal,auroc,auroc_lo,auroc_hi,norm_recall@5% (10%),std (10%),norm_recall@5% (20%),std (20%),beats_heuristic
kev_finetuned,keep_or_drop,0.735,0.692,0.780,0.443,0.168,0.712,0.096,False
jev,keep_or_drop,0.788,0.744,0.829,0.641,0.168,0.899,0.083,False
heuristic_logreg,logreg,0.701,0.655,0.747,0.585,0.160,0.813,0.057,
heuristic_few_evidence,few_evidence,0.544,0.506,0.584,0.205,0.137,0.314,0.185,


## 8. Head to head

The summary gives each model's score with its own spread. This table asks the sharper question: on the **same** traces, how often and by how much does one model beat another?

- `auroc_diff`: A's AUROC minus B's, with a 95% interval from resampling the test portion. An interval that includes 0 means the test portion cannot tell them apart.
- `norm_recall_diff`: A minus B at the 5% budget, mean ± standard deviation over the rebalanced test sets. Both models are scored on the same sets, so this is a paired difference.
- `wins`: in how many of the `N_RESAMPLES` sets A did strictly better than B.

In [ ]:
def paired_recall(scores_a, scores_b, br, rate=SAMPLE_RATE):
    a = np.array([sampler_metrics(scores_a[idx], y_test[idx], rate)["normalized_recall"] for idx in RESAMPLES[br]])
    b = np.array([sampler_metrics(scores_b[idx], y_test[idx], rate)["normalized_recall"] for idx in RESAMPLES[br]])
    return a - b

pairs = []
for key_a, key_b in itertools.combinations(ROW_KEYS, 2):
    if key_a in REFERENCE and key_b in REFERENCE:
        continue
    scores_a, scores_b = series_of(key_a)[2], series_of(key_b)[2]
    diffs = np.array([roc_auc_score(y_test[idx], scores_a[idx]) - roc_auc_score(y_test[idx], scores_b[idx]) for idx in BOOT])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    row = {"A": key_a, "B": key_b,
           "auroc_diff": roc_auc_score(y_test, scores_a) - roc_auc_score(y_test, scores_b),
           "auroc_diff_lo": lo, "auroc_diff_hi": hi,
           "auroc_differs": bool(lo > 0 or hi < 0)}
    for br in TARGET_BASE_RATES:
        d = paired_recall(scores_a, scores_b, br)
        row[f"norm_recall_diff ({br:.0%})"] = d.mean()
        row[f"std ({br:.0%})"] = d.std()
        row[f"wins ({br:.0%})"] = f"{int((d > 0).sum())}/{len(d)}"
    pairs.append(row)

head_to_head = pd.DataFrame(pairs)
head_to_head.style.format(precision=3).hide(axis="index")

## 9. Where they differ

Two views that a single number hides.

**By failure type.** The dataset marks how each wrong trace came about: `natural` (the agent simply got it wrong) or one of the seeded faults (`contradiction`, `missing_evidence`, `early_stop`, `weak_model`). For each type, the AUROC of telling that type's wrong traces apart from all correct traces. A model can look strong overall by catching only the faults that leave obvious marks.

**Overlap of picks.** For each pair, the share of traces both would send for review among those either would, at the 5% budget on the full test portion. Low overlap between two good models means combining them could catch more.

In [11]:
failure_type = np.array([t.get("failure_type", "unknown") for t in test])
types = sorted(set(failure_type[y_test]))
correct = ~y_test

by_type = []
for key in ROW_KEYS:
    scores = series_of(key)[2]
    row = {"model": key}
    for ft in types:
        mask = correct | (y_test & (failure_type == ft))
        row[f"{ft} (n={int((y_test & (failure_type == ft)).sum())})"] = roc_auc_score(y_test[mask], scores[mask])
    by_type.append(row)
by_failure_type = pd.DataFrame(by_type)
print("AUROC of each failure type's wrong traces against all correct traces (test portion)")
print(by_failure_type.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

k = max(1, math.ceil(SAMPLE_RATE * len(y_test)))
tie_break = np.random.default_rng(SEED).permutation(len(y_test))   # rules have many ties; break them the same way for all
def picks(key):
    scores = series_of(key)[2][tie_break]
    return set(tie_break[np.argsort(-scores, kind="stable")[:k]])

overlap = pd.DataFrame(
    [[len(picks(a) & picks(b)) / len(picks(a) | picks(b)) for b in ROW_KEYS] for a in ROW_KEYS],
    index=ROW_KEYS, columns=ROW_KEYS,
)
print(f"\nshare of picks in common at the {SAMPLE_RATE:.0%} budget ({k} of {len(y_test)} test traces each)")
print(overlap.to_string(float_format=lambda v: f"{v:.2f}"))

AUROC of each failure type's wrong traces against all correct traces (test portion)
                 model  contradiction (n=17)  early_stop (n=21)  missing_evidence (n=30)  natural (n=162)  weak_model (n=32)
         kev_finetuned                 0.756              0.682                    0.775            0.730              0.745
                   jev                 0.810              0.715                    0.882            0.784              0.759
      heuristic_logreg                 0.586              0.787                    0.724            0.664              0.875
heuristic_few_evidence                 0.557              0.619                    0.554            0.519              0.611

share of picks in common at the 5% budget (23 of 456 test traces each)
                        kev_finetuned  jev  heuristic_logreg  heuristic_few_evidence
kev_finetuned                    1.00 0.05              0.28                    0.21
jev                              0.05 1.00       

## 10. Calibration

Ranking is what the sampler needs, but the engine's thresholds and alerts read the probability itself. For each model, one temperature is fitted on the calibration portion and ECE is reported on the test portion, before and after. Lower is better; a temperature far from 1 means the model's raw probabilities are over- or under-confident on these traces.

Skipped when `CALIBRATE` is off.

In [12]:
def fit_temperature(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    logits = np.log(p / (1 - p))
    def nll(T):
        q = 1 / (1 + np.exp(-logits / T))
        q = np.clip(q, 1e-9, 1 - 1e-9)
        return -np.mean(y * np.log(q) + (1 - y) * np.log(1 - q))
    return minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x

def ece(p, y, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return sum(abs(p[idx == b].mean() - y[idx == b].mean()) * (idx == b).mean()
               for b in range(bins) if (idx == b).any())

def apply_temperature(p, T):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return 1 / (1 + np.exp(-np.log(p / (1 - p)) / T))

calibration_rows = []
if CALIBRATE:
    for name in AVAILABLE:
        for signal in SIGNAL_READ:
            p_calib, p_test = signal_of(name, signal, "calibration"), signal_of(name, signal)
            T = fit_temperature(p_calib, y_calib.astype(float))
            calibration_rows.append({
                "model": name, "signal": signal, "temperature": float(T),
                "ece_before": float(ece(p_test, y_test)), "ece_after": float(ece(apply_temperature(p_test, T), y_test)),
                "hit_bound": bool(not 0.06 < T < 19.9),
            })
calibration = pd.DataFrame(calibration_rows)
if calibration.empty:
    print("CALIBRATE is off: no temperatures fitted")
else:
    print(f"temperatures fitted on {len(calib)} calibration traces, ECE on {len(test)} test traces")
    print(calibration.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    if calibration.hit_bound.any():
        print("a temperature at the search bound (hit_bound) means that signal is close to uninformative for that model")

temperatures fitted on 444 calibration traces, ECE on 456 test traces
        model       signal  temperature  ece_before  ece_after  hit_bound
kev_finetuned answer_wrong        1.504       0.083      0.061      False
kev_finetuned keep_or_drop        1.616       0.063      0.049      False
          jev answer_wrong        2.384       0.161      0.125      False
          jev keep_or_drop        4.466       0.157      0.090      False


## 11. Latency and cost

Round-trip time per request as this notebook saw it, on the test portion. The numbers are not like for like: the local models run on this machine with no network in between, and Jev's include the trip to TypeSafe's servers. Answers reused from the cache keep the latency of the call that produced them.

`input_tokens` is what each server reported reading. Each model counts with its own tokenizer, and Laya counts one row per question, so compare it across models only loosely. For Jev it is what the bill is based on.

In [13]:
cost_rows = []
for name in AVAILABLE:
    records = SCORES[name]["test"]
    latency = np.array([r["latency_ms"] for r in records])
    tokens = [r["input_tokens"] for r in records if r.get("input_tokens") is not None]
    scored = sum(len(v) for v in SCORES[name].values())
    cost_rows.append({
        "model": name, "where": "hosted" if MODELS[name].get("api_key_env") else "local",
        "latency_p50_ms": float(np.percentile(latency, 50)), "latency_p95_ms": float(np.percentile(latency, 95)),
        "input_tokens_per_trace": float(np.mean(tokens)) if tokens else float("nan"),
        "traces_scored": scored,
        "input_tokens_total": int(sum(r["input_tokens"] or 0 for v in SCORES[name].values() for r in v)),
    })
latency_cost = pd.DataFrame(cost_rows)
print(latency_cost.to_string(index=False, float_format=lambda v: f"{v:.0f}"))

        model  where  latency_p50_ms  latency_p95_ms  input_tokens_per_trace  traces_scored  input_tokens_total
kev_finetuned  local             103             145                     481            900              432859
          jev hosted             734             927                     772            900              693953


## 12. Full results table and saved files

Every model, signal, base rate and sample rate, in the fine-tuning notebooks' format, so rows can be set side by side with theirs. Everything is also written to `runs/observer-comparison/`.

In [14]:
results = results_table()
results.to_csv(WORK_DIR / "results.csv", index=False)
summary.to_csv(WORK_DIR / "summary.csv", index=False)
head_to_head.to_csv(WORK_DIR / "head_to_head.csv", index=False)

comparison = {
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "seed": SEED,
    "decision_signal": DECISION_SIGNAL,
    "models": {name: {k: v for k, v in MODELS[name].items() if k != "api_key_env"} for name in AVAILABLE},
    "skipped": SKIPPED,
    "test": {"traces": len(test), "base_rate": round(float(y_test.mean()), 4)},
    "calibration": {"traces": len(calib), "used": CALIBRATE},
    "evaluation": {"target_base_rates": TARGET_BASE_RATES, "sample_rates": SAMPLE_RATES,
                   "n_resamples": N_RESAMPLES, "n_bootstrap": int(len(BOOT))},
    "success_bar": {str(br): bar[br] for br in TARGET_BASE_RATES},
    "summary": summary.to_dict("records"),
    "head_to_head": head_to_head.to_dict("records"),
    "by_failure_type": by_failure_type.to_dict("records"),
    "pick_overlap": overlap.to_dict(),
    "calibration_fit": calibration.to_dict("records"),
    "latency_cost": latency_cost.to_dict("records"),
    "results": results.to_dict("records"),
}
(WORK_DIR / "comparison.json").write_text(json.dumps(comparison, indent=2, default=float))
print(f"{len(results)} result rows; wrote results.csv, summary.csv, head_to_head.csv and comparison.json to {WORK_DIR}")
results.style.format(precision=3).hide(axis="index")

66 result rows; wrote results.csv, summary.csv, head_to_head.csv and comparison.json to /Users/samuelabolo/kode/hobby/observer-ai/finetune/notebooks/runs/observer-comparison


model,signal,base_rate,sample_rate,n_rebalanced,auroc_full_test,recall_mean,recall_std,oracle_recall,random_recall,norm_recall_mean,norm_recall_std
heuristic,bad_stop,0.100,0.050,216,0.502,0.050,0.052,0.500,0.051,-0.002,0.115
heuristic,few_evidence,0.100,0.050,216,0.544,0.143,0.061,0.500,0.051,0.205,0.137
heuristic,logreg,0.100,0.050,216,0.701,0.314,0.072,0.500,0.051,0.585,0.160
heuristic,low_overlap,0.100,0.050,216,0.613,0.052,0.046,0.500,0.051,0.003,0.103
heuristic,no_search,0.100,0.050,216,0.536,0.125,0.075,0.500,0.051,0.165,0.166
heuristic,short_answer,0.100,0.050,216,0.357,0.036,0.040,0.500,0.051,-0.032,0.088
heuristic,tool_error,0.100,0.050,216,0.522,0.084,0.054,0.500,0.051,0.074,0.121
jev,answer_wrong,0.100,0.050,216,0.808,0.382,0.083,0.500,0.051,0.737,0.185
jev,keep_or_drop,0.100,0.050,216,0.788,0.341,0.074,0.500,0.051,0.646,0.165
kev_finetuned,answer_wrong,0.100,0.050,216,0.730,0.250,0.074,0.500,0.051,0.443,0.165
